# 01 - Datos, preprocesamiento y modelo baseline (TensorFlow/Keras)

**Objetivo:** preparar MNIST, revisar la distribución de clases y establecer una línea base con Dev explícito.

Configuración de la guía: Train/Dev/Test ≈ 48.000/12.000/10.000; normalización de 0-255 a 0-1; Dense(128, ReLU) y salida de 10 clases Softmax; Adam 0.001; batch 64; 5 épocas.

## 1. Importar herramientas y fijar la semilla

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers, callbacks
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("TensorFlow:", tf.__version__)

## 2. Cargar, separar y normalizar MNIST

In [ ]:
(x_all, y_all), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_dev, y_train, y_dev = train_test_split(
    x_all, y_all, test_size=0.20, random_state=SEED, stratify=y_all
)
x_train = x_train.astype("float32") / 255.0
x_dev = x_dev.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
print("Train:", x_train.shape, y_train.shape)
print("Dev:", x_dev.shape, y_dev.shape)
print("Test:", x_test.shape, y_test.shape)
print("Rango de píxeles:", x_train.min(), x_train.max())

## 3. Distribución de los diez dígitos

In [ ]:
def distribucion(y, nombre):
    conteos = np.bincount(y, minlength=10)
    tabla = pd.DataFrame({"digito": np.arange(10), "cantidad": conteos,
                          "porcentaje": np.round(conteos / len(y) * 100, 2)})
    print(nombre)
    display(tabla)
    return tabla

dist_train = distribucion(y_train, "TRAIN")
dist_dev = distribucion(y_dev, "DEV")
dist_test = distribucion(y_test, "TEST")

fig, ax = plt.subplots(figsize=(8, 4))
for nombre, y in [("Train", y_train), ("Dev", y_dev), ("Test", y_test)]:
    ax.plot(np.arange(10), np.bincount(y, minlength=10) / len(y) * 100, marker="o", label=nombre)
ax.set(xlabel="Dígito", ylabel="Porcentaje (%)", title="Distribución de clases")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

**Análisis:** confirma que aparecen los diez dígitos y que los porcentajes entre Train, Dev y Test son semejantes. Escribe una conclusión a partir de las tablas.

## 4. Arquitectura baseline y entrenamiento

In [ ]:
def crear_baseline():
    modelo = keras.Sequential([
        layers.Input(shape=(28, 28)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                   loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return modelo

keras.backend.clear_session()
tf.random.set_seed(SEED)
modelo_base = crear_baseline()
modelo_base.summary()
hist_base = modelo_base.fit(x_train, y_train, validation_data=(x_dev, y_dev),
                            epochs=5, batch_size=64, verbose=1)
dev_loss, dev_acc = modelo_base.evaluate(x_dev, y_dev, verbose=0)
print(f"Dev loss: {dev_loss:.4f} | Dev accuracy: {dev_acc:.4f}")

## 5. Curvas de aprendizaje

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_base.history["loss"], label="Train")
ax.plot(hist_base.history["val_loss"], label="Dev")
ax.set(xlabel="Época", ylabel="Loss", title="Baseline - Loss")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist_base.history["accuracy"], label="Train")
ax.plot(hist_base.history["val_accuracy"], label="Dev")
ax.set(xlabel="Época", ylabel="Accuracy", title="Baseline - Accuracy")
ax.grid(True, alpha=.3); ax.legend(); plt.show()

**En el informe:** incluye las formas de los datos, la distribución, `model.summary()` y ambas gráficas. Compara Train y Dev; si Train mejora y Dev se estanca o empeora, puede ser una señal de overfitting. No uses Test para escoger configuraciones.